In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=OPENROUTER_API_KEY,
)

# First API call with reasoning
response = client.chat.completions.create(
  model="qwen/qwen3.8-27b:free",
  messages=[
          {
            "role": "user",
            "content": "How many r's are in the word 'strawberry'?"
          }
        ],
  extra_body={"reasoning": {"enabled": True}}
)

# Extract the assistant message with reasoning_details
response = response.choices[0].message
print(response.content)
# Preserve the assistant message with reasoning_details
messages = [
  {"role": "user", "content": "How many r's are in the word 'strawberry'?"},
  {
    "role": "assistant",
    "content": response.content,
    "reasoning_details": response.reasoning_details  # Pass back unmodified
  },
  {"role": "user", "content": "Are you sure? Think carefully."}
]

# Second API call - model continues reasoning from where it left off
response2 = client.chat.completions.create(
  model="qwen/qwen3.8-27b:free",
  messages=messages,
  extra_body={"reasoning": {"enabled": True}}
)

The word **“strawberry”** has **3 r’s**.


In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
CLOUDFLARE_API_KEY = os.getenv("CLOUDFLARE_API_KEY-1")
# You will also need your Account ID to build the URL
CLOUDFLARE_ACCOUNT_ID = os.getenv("CLOUDFLARE_ACCOUNT_ID-1")

# Initialize the client but point it at Cloudflare instead of OpenAI
client = OpenAI(
    api_key=CLOUDFLARE_API_KEY,
    base_url=f"https://api.cloudflare.com/client/v4/accounts/{CLOUDFLARE_ACCOUNT_ID}/ai/v1"
)

print("Sending request to Cloudflare Workers AI...")

# Use standard OpenAI syntax
response = client.chat.completions.create(
    model="@cf/openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are an expert Python developer."},
        {"role": "user", "content": "Write a quick Python script to sort an array."}
    ]
)

# Print the response
print("\n--- AI RESPONSE ---")
print(response.choices[0].message.content)

Sending request to Cloudflare Workers AI...

--- AI RESPONSE ---
Below is a **stand‑alone Python script** that reads a list of numbers (or strings) from the command line, sorts it, and prints the sorted result.  
It demonstrates both the built‑in sorting methods and a simple manual implementation (bubble‑sort) so you can see how sorting works under the hood.

```python
#!/usr/bin/env python3
"""
quick_sorter.py – a tiny utility to sort an array.

Usage examples
--------------
$ python quick_sorter.py 5 2 9 1 3
Sorted (built‑in): [1, 2, 3, 5, 9]

$ python quick_sorter.py apple banana cherry
Sorted (built‑in): ['apple', 'banana', 'cherry']

$ python quick_sorter.py --manual 5 2 9 1 3
Sorted (bubble‑sort): [1, 2,


In [2]:
# ============================================================
# SETUP — Run this once
# ============================================================

import os
import re
import json
from collections import Counter, defaultdict
from dotenv import load_dotenv

load_dotenv()

from pinecone import Pinecone
from google import genai
from google.genai import types

# --- Config (adjust if needed) ---
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY   = os.getenv("GEMINI_API_KEY")
INDEX_NAME       = os.getenv("INDEX_NAME", "studybuddy")
USER_ID          = "user_1791645587"
EMBEDDING_DIM    = 768
EMBEDDING_MODEL  = "gemini-embedding-2"

# --- Clients ---
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

gemini = genai.Client(api_key=GEMINI_API_KEY)

def embed(text: str) -> list[float]:
    """Embed a single string with Gemini Embedding 2."""
    r = gemini.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=[text],
        config=types.EmbedContentConfig(output_dimensionality=EMBEDDING_DIM),
    )
    return r.embeddings[0].values

# --- Pull ALL chunks for this user once, reuse everywhere ---
print("📥 Fetching all chunks...")
all_chunks = index.query(
    vector=[0.01] * EMBEDDING_DIM,
    top_k=10000,
    namespace="notes",
    filter={"user_id": {"$eq": USER_ID}},
    include_metadata=True,
).matches

print(f"✅ Retrieved {len(all_chunks)} chunks for user {USER_ID}")

# Helper: get text safely
def chunk_text(m):
    return (m.metadata or {}).get("text", "")

📥 Fetching all chunks...
✅ Retrieved 141 chunks for user user_1791645587


In [3]:
# ============================================================
# TEST A — Chunk distribution per module
# ============================================================

# Keyword signatures per module (from the PDF syllabus)
MODULE_KEYWORDS = {
    "Module I   (Intro: alphabets, strings, languages)": [
        "alphabet", "formal language", "kleene", "concatenation",
        "word over", "chomsky hierarchy"
    ],
    "Module II  (Finite Automata, DFA, NFA, ε-NFA)": [
        "deterministic finite", "dfa", "nondeterministic finite", "nfa",
        "ε-transition", "epsilon", "transition function", "accept state"
    ],
    "Module III (Regular Expressions, Pumping Lemma)": [
        "regular expression", "regexp", "pumping lemma",
        "closure propert", "minimization", "dfa minimization"
    ],
    "Module IV  (Context-Free Grammars, Parse Trees)": [
        "context-free grammar", "cfg", "parse tree", "derivation",
        "leftmost", "rightmost", "ambiguous grammar", "yield of a parse"
    ],
    "Module V   (Pushdown Automata)": [
        "pushdown automaton", "pda", "stack", "acceptance by final state",
        "acceptance by empty stack", "instantaneous description"
    ],
    "Module VI  (CFL Properties, CYK, Ogden)": [
        "chomsky normal form", "cnf", "cyk", "cocke", "ogden",
        "decision propert", "parikh"
    ],
    "Module VII (Turing Machines)": [
        "turing machine", "halting", "tape", "instantaneous description",
        "universal turing", "church-turing"
    ],
    "Module VIII(Undecidability, PCP, Diagonalization)": [
        "undecidable", "diagonalization", "post's correspondence", "pcp",
        "recursively enumerable", "godel", "cantor"
    ],
}

def classify_chunk(text: str) -> str:
    """Assign a chunk to the module with the most keyword hits."""
    text_l = text.lower()
    scores = {m: sum(1 for kw in kws if kw in text_l)
              for m, kws in MODULE_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "UNCLASSIFIED"

# Classify every chunk
module_counts = Counter()
module_examples = defaultdict(list)

for m in all_chunks:
    txt = chunk_text(m)
    mod = classify_chunk(txt)
    module_counts[mod] += 1
    if len(module_examples[mod]) < 2:
        module_examples[mod].append(txt[:200])

# Print report
print("=" * 70)
print("CHUNK DISTRIBUTION BY MODULE")
print("=" * 70)

total = sum(module_counts.values())
for mod in sorted(MODULE_KEYWORDS.keys()):
    c = module_counts.get(mod, 0)
    pct = (c / total * 100) if total else 0
    bar = "█" * int(pct / 2)
    print(f"{mod:<58} {c:>4} ({pct:5.1f}%) {bar}")

if "UNCLASSIFIED" in module_counts:
    c = module_counts["UNCLASSIFIED"]
    print(f"{'UNCLASSIFIED':<58} {c:>4} ({c/total*100:5.1f}%)")

print("=" * 70)
print(f"TOTAL: {total} chunks")

# Sample text per module (so you can eyeball classification quality)
print("\n--- Sample chunks per module ---")
for mod in sorted(module_examples.keys()):
    print(f"\n[{mod}]")
    for i, ex in enumerate(module_examples[mod], 1):
        print(f"  {i}. {ex[:180]}...")

CHUNK DISTRIBUTION BY MODULE
Module I   (Intro: alphabets, strings, languages)            26 ( 18.4%) █████████
Module II  (Finite Automata, DFA, NFA, ε-NFA)                27 ( 19.1%) █████████
Module III (Regular Expressions, Pumping Lemma)              12 (  8.5%) ████
Module IV  (Context-Free Grammars, Parse Trees)              17 ( 12.1%) ██████
Module V   (Pushdown Automata)                                7 (  5.0%) ██
Module VI  (CFL Properties, CYK, Ogden)                       3 (  2.1%) █
Module VII (Turing Machines)                                 16 ( 11.3%) █████
Module VIII(Undecidability, PCP, Diagonalization)             1 (  0.7%) 
UNCLASSIFIED                                                 32 ( 22.7%)
TOTAL: 141 chunks

--- Sample chunks per module ---

[Module I   (Intro: alphabets, strings, languages)]
  1. In mathematical logic, a Godel numbering is a function that assigns to each symbol and wellformed formula of some formal language a unique natural number, calle

In [4]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("studybuddy")

# Fetch all chunks for the user
results = index.query(
    vector=[0.0] * 768,
    top_k=10000,
    include_metadata=True,
    namespace="notes",
    filter={"user_id": {"$eq": "user_1791642192"}},
)

chunks = [m.metadata.get("text", "") for m in results.matches]
print(f"Total chunks: {len(chunks)}")
print("=" * 70)

# Check for corruption markers
import re
corruption_patterns = {
    "fused_words": r'[a-z][A-Z]',           # "aRegular"
    "truncated": r'\b(composi|aregular|th|calle|disad)\b',
    "mid_word_break": r'\bcts\b|\bosition\b',
    "page_numbers": r'^\d{1,3}$',
}

print("CORRUPTION CHECK:")
for name, pattern in corruption_patterns.items():
    hits = sum(1 for c in chunks if re.search(pattern, c, re.MULTILINE))
    print(f"  {name}: {hits}/{len(chunks)} chunks")

print("=" * 70)
print("\nFIRST 3 CHUNKS (raw text):\n")
for i, c in enumerate(chunks[:3], 1):
    print(f"--- Chunk {i} ({len(c)} chars) ---")
    print(c[:400])
    print()

Total chunks: 0
CORRUPTION CHECK:
  fused_words: 0/0 chunks
  truncated: 0/0 chunks
  mid_word_break: 0/0 chunks
  page_numbers: 0/0 chunks

FIRST 3 CHUNKS (raw text):



In [5]:
broken_symbol_chunks = [c for c in chunks if '�' in c]
print(f"Chunks with broken glyphs: {len(broken_symbol_chunks)}")
print(f"Percentage: {len(broken_symbol_chunks)/len(chunks)*100:.1f}%")

# Which modules are affected?
for c in broken_symbol_chunks[:5]:
    print(f"--- {c[:200]}")

Chunks with broken glyphs: 0


ZeroDivisionError: division by zero

In [6]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("studybuddy")

# List all users
users_resp = index.query(
    vector=[0.0] * 768,
    top_k=10000,
    include_metadata=True,
    namespace="users",
    filter={"type": {"$eq": "user_auth"}},
)

print("ALL USERS IN PINECONE:")
print("=" * 60)
for m in users_resp.matches:
    md = m.metadata or {}
    uid = md.get("user_id")
    email = md.get("email")
    if uid:
        print(f"  {email}  →  {uid}")
print("=" * 60)

# For each user, count their note chunks
print("\nCHUNK COUNTS PER USER:")
print("=" * 60)
for m in users_resp.matches:
    md = m.metadata or {}
    uid = md.get("user_id")
    email = md.get("email")
    if not uid:
        continue
    
    notes_resp = index.query(
        vector=[0.0] * 768,
        top_k=10000,
        include_metadata=True,
        namespace="notes",
        filter={"user_id": {"$eq": uid}, "type": {"$eq": "notes"}},
    )
    print(f"  {email} ({uid}): {len(notes_resp.matches)} chunks")

ALL USERS IN PINECONE:
  rabiabde0@gmail.com  →  user_1791645587

CHUNK COUNTS PER USER:
  rabiabde0@gmail.com (user_1791645587): 140 chunks
